In [26]:

GITHUB_USER = 'Skm48'   # GitHub username
REPO_NAME   = 'MINI_PROJECT'

import os

if not os.path.exists(f'/content/{REPO_NAME}'):
    # First time — clone
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git
    print('Repo cloned!')
else:
    # Already cloned — pull latest
    !cd /content/{REPO_NAME} && git pull
    print('Pulled latest changes!')

os.chdir(f'/content/{REPO_NAME}')
print(f'Working directory: {os.getcwd()}')


Already up to date.
Pulled latest changes!
Working directory: /content/MINI_PROJECT


In [23]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
!cp /content/drive/MyDrive/Collab\ Files/Dataset.zip /content/MINI_PROJECT/data/
!unzip -q /content/MINI_PROJECT/data/Dataset.zip -d /content/MINI_PROJECT/data/

replace /content/MINI_PROJECT/data/chest_xray/test/NORMAL/IM-0001-0001.jpeg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [25]:
# Remove the Mac junk folder and nested duplicate
!rm -rf /content/MINI_PROJECT/data/chest_xray/__MACOSX
!rm -rf /content/MINI_PROJECT/data/chest_xray/chest_xray

# Check it's clean
!ls /content/MINI_PROJECT/data/chest_xray/

test  train  val


In [27]:

import os
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split


In [28]:
# ──────────────────────────────────────────────
# 1. Collect all image paths + labels
# ──────────────────────────────────────────────

def collect_image_paths(raw_dir: str) -> pd.DataFrame:
    records = []
    label_map = {"NORMAL": 0, "PNEUMONIA": 1}

    for split in ["train", "val", "test"]:
        split_dir = os.path.join(raw_dir, split)
        if not os.path.exists(split_dir):
            print(f"Warning: {split_dir} not found, skipping")
            continue

        for class_name in ["NORMAL", "PNEUMONIA"]:
            class_dir = os.path.join(split_dir, class_name)
            if not os.path.exists(class_dir):
                continue

            for fname in os.listdir(class_dir):
                if fname.lower().endswith((".jpeg", ".jpg", ".png")):
                    records.append({
                        "filepath": os.path.join(class_dir, fname),
                        "label": label_map[class_name],
                        "original_split": split,
                    })

    df = pd.DataFrame(records)
    print(f"Total images found: {len(df)}")
    print(f"  By original split: {dict(df['original_split'].value_counts())}")
    print(f"  By label: {dict(df['label'].value_counts())}")
    return df


In [29]:
df = collect_image_paths("data/chest_xray")
df.head()

Total images found: 5856
  By original split: {'train': np.int64(5216), 'test': np.int64(624), 'val': np.int64(16)}
  By label: {1: np.int64(4273), 0: np.int64(1583)}


,filepath,label,original_split
0,data/chest_xray/train/NORMAL/NORMAL2-IM-1422-0...,0,train
1,data/chest_xray/train/NORMAL/IM-0147-0001.jpeg,0,train
2,data/chest_xray/train/NORMAL/NORMAL2-IM-0939-0...,0,train
3,data/chest_xray/train/NORMAL/NORMAL2-IM-1287-0...,0,train
4,data/chest_xray/train/NORMAL/IM-0262-0001.jpeg,0,train


In [30]:
from sklearn.model_selection import train_test_split

def stratified_split(df, seed=42, save_path=None):
    # Keep original test set untouched
    test_df = df[df["original_split"] == "test"].copy()

    # Merge train + val (val is only 16 images — useless)
    trainval_df = df[df["original_split"].isin(["train", "val"])].copy()

    print(f"Merged train+val: {len(trainval_df)} images")
    print(f"Original test kept: {len(test_df)} images")

    # Split trainval into new train (80%) and val (10%)
    # val_ratio = 0.1 / (0.8 + 0.1) ≈ 0.111
    train_df, val_df = train_test_split(
        trainval_df,
        test_size=0.111,
        stratify=trainval_df["label"],
        random_state=seed,
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print(f"\nNew split:")
    print(f"  Train: {len(train_df)}  (Normal: {sum(train_df['label']==0)}, Pneumonia: {sum(train_df['label']==1)})")
    print(f"  Val:   {len(val_df)}  (Normal: {sum(val_df['label']==0)}, Pneumonia: {sum(val_df['label']==1)})")
    print(f"  Test:  {len(test_df)}  (Normal: {sum(test_df['label']==0)}, Pneumonia: {sum(test_df['label']==1)})")

    # Save for reproducibility
    if save_path:
        train_df["split"] = "train"
        val_df["split"] = "val"
        test_df["split"] = "test"
        all_splits = pd.concat([train_df, val_df, test_df], ignore_index=True)
        all_splits.to_csv(save_path, index=False)
        print(f"Split indices saved to: {save_path}")
        train_df = train_df.drop(columns=["split"])
        val_df = val_df.drop(columns=["split"])
        test_df = test_df.drop(columns=["split"])

    return train_df, val_df, test_df

In [31]:
train_df, val_df, test_df = stratified_split(df, save_path="data/split_indices.csv")

Merged train+val: 5232 images
Original test kept: 624 images

New split:
  Train: 4651  (Normal: 1199, Pneumonia: 3452)
  Val:   581  (Normal: 150, Pneumonia: 431)
  Test:  624  (Normal: 234, Pneumonia: 390)
Split indices saved to: data/split_indices.csv


In [32]:
from torchvision import transforms

def get_transforms(image_size=224):
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]

    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    return {"train": train_transform, "val": eval_transform, "test": eval_transform}

In [33]:
from PIL import Image

tfms = get_transforms()
img = Image.open(train_df.iloc[0]["filepath"]).convert("RGB")
tensor = tfms["train"](img)
print(f"Original size: {img.size}")
print(f"Tensor shape: {tensor.shape}")
print(f"Pixel range: [{tensor.min():.2f}, {tensor.max():.2f}]")


Original size: (976, 672)
Tensor shape: torch.Size([3, 224, 224])
Pixel range: [-2.12, 1.87]


In [34]:
class ChestXrayDataset(Dataset):
    """
    PyTorch Dataset for chest X-ray images.

    Args:
        dataframe: DataFrame with 'filepath' and 'label' columns
        transform: torchvision transform to apply
    """

    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(row["label"], dtype=torch.long)
        return image, label

In [35]:
train_ds = ChestXrayDataset(train_df, transform=tfms["train"])
img, label = train_ds[0]
print(f"Dataset size: {len(train_ds)}")
print(f"Image shape: {img.shape}")
print(f"Label: {label}")

Dataset size: 4651
Image shape: torch.Size([3, 224, 224])
Label: 1


In [39]:
def compute_class_weights(train_df):
    counts = Counter(train_df["label"].values)
    total = sum(counts.values())
    weights = [total / (len(counts) * counts[i]) for i in range(len(counts))]
    weights_tensor = torch.FloatTensor(weights)
    print(f"Class weights: Normal={weights[0]:.3f}, Pneumonia={weights[1]:.3f}")
    return weights_tensor
compute_class_weights(train_df)

Class weights: Normal=1.940, Pneumonia=0.674


tensor([1.9395, 0.6737])

In [42]:
!cd /content/MINI_PROJECT && git config user.name "Skm48"
!cd /content/MINI_PROJECT && git config user.email "skm48@student.le.ac.uk"

In [44]:
!cd /content/MINI_PROJECT && git remote set-url origin https://Skm48@github.com/Skm48/MINI_PROJECT.git

In [48]:
!cp /content/drive/MyDrive/Colab\ Notebooks/01_setup_EDA_Test.ipynb /content/MINI_PROJECT/notebooks/